# Занятие 32. Практика: решающее дерево — школьный пропуск

Вы пишете код в ячейках заданий. Блоки **«Легенда»** и **«Дано»** не меняйте.

Главная модель — **DecisionTreeClassifier** (решающее дерево). Теория — занятие 31, ноутбук `Урок_31_Решающее_дерево.ipynb`.

На вымышленном датасете школьного турникета сравним глубокие и ограниченные деревья, подберём `max_depth`, посмотрим `plot_tree`, `min_samples_leaf`, обрезку `ccp_alpha`, матрицу ошибок и важности признаков.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Split и импорты | 3 |
| 2 | Глубина и переобучение | 4 |
| 3 | Подбор max_depth и график | 4 |
| 4 | Визуализация дерева (`plot_tree`) | 3 |
| 5 | `min_samples_leaf` | 4 |
| 6 | Обрезка `ccp_alpha` | 4 |
| 7 | Confusion matrix (heatmap) | 3 |
| 8 | Feature importance | 3 |
| 9 | Итог | 2 |
| | **Итого** | **30** |


---
## Легенда: турникет лицея «Северный маяк»

Вы помогаете **вымышленному** лицею настроить черновик правил для электронного турникета у входа.

Каждое утро система смотрит на несколько признаков и предлагает решение: **пустить** ученика в здание или **не пустить** (отправить к дежурному за устным решением). Это **не оценка личности** и не «хороший / плохой ученик» — только учебный пример про правила и дерево решений.

### Признаки (таблица)

| Признак | Смысл |
|---------|--------|
| `late_min` | опоздание в минутах (0…40) |
| `has_note` | есть справка / записка от родителя (0 или 1) |
| `weekday` | день недели (0 = понедельник … 6 = воскресенье) |
| `shift` | смена (0 — первая, 1 — вторая) |
| `temp_c` | температура на улице, °C |
| `locker_id` | «шумный» признак: номер шкафчика (на решение почти не влияет) |

### Метка

| Значение | Смысл |
|----------|--------|
| `1` | пустить |
| `0` | не пустить (нужна ручная проверка у дежурного) |

Данные **синтетические**: мы сами задали скрытый «устав», добавили немного случайных сбоев. Ваша задача — обучить дерево, которое восстановит понятные правила, и не дать модели запомнить шум.

### Короткий словарь

| Слово | Значение |
|-------|----------|
| **признак** | одно число в таблице (например, минуты опоздания) |
| **метка / класс** | правильный ответ: пустить или нет |
| **train** | обучающая выборка — на ней дерево **учится** |
| **validation** | проверочная — на ней **сравниваем** варианты настроек |
| **переобучение** | модель слишком хорошо запомнила train и хуже работает на новых данных |
| **`max_depth`** | максимальная глубина дерева |
| **`min_samples_leaf`** | минимальное число объектов в листе |
| **`ccp_alpha`** | сила обрезки (pruning) после обучения |
| **`plot_tree`** | рисунок дерева как «устав» из вопросов «если… то…» |


---
## Дано: синтетический журнал пропусков

Ячейку ниже **не меняйте**. Она создаёт таблицу признаков `X`, метки `y` и список имён `FEATURE_NAMES`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n_samples = 600

late_min = rng.integers(0, 41, size=n_samples)
has_note = rng.integers(0, 2, size=n_samples)
weekday = rng.integers(0, 7, size=n_samples)
shift = rng.integers(0, 2, size=n_samples)
temp_c = rng.normal(loc=5.0, scale=10.0, size=n_samples).round(1)
locker_id = rng.integers(1, 201, size=n_samples)

# Скрытый «устав» лицея (для генерации меток; модель его не видит напрямую):
# 1) есть справка → почти всегда пустить;
# 2) опоздание ≤ 8 мин → пустить;
# 3) опоздание ≤ 15 мин и не пн/вт → пустить;
# 4) иначе → не пустить. Плюс редкие случайные сбои журнала.
admit = np.zeros(n_samples, dtype=int)
admit[(has_note == 1) | (late_min <= 8) | ((late_min <= 15) & (weekday >= 2))] = 1
flip = rng.random(n_samples) < 0.08
admit = np.where(flip, 1 - admit, admit)

FEATURE_NAMES = [
    "late_min",
    "has_note",
    "weekday",
    "shift",
    "temp_c",
    "locker_id",
]
CLASS_NAMES = ["не пустить", "пустить"]

X = np.column_stack([late_min, has_note, weekday, shift, temp_c, locker_id])
y = admit

print("Объектов:", len(X))
print("Классы [не пустить, пустить]:", np.bincount(y))
pd.DataFrame(X, columns=FEATURE_NAMES).head()


---
## Задание 1. Split и импорты — **3 балла**

Разделите данные на **train** и **validation**.

1. Задайте `RANDOM_STATE = 42` (если ещё не задан).
2. Вызовите `train_test_split` с `test_size=0.30` и `stratify=y`.
3. В sklearn параметр называется `test_size`, но отложенную часть в этой практике используйте как **validation**: по ней будете сравнивать деревья.
4. Сохраните массивы в переменные `X_train`, `X_val`, `y_train`, `y_val`.
5. Напечатайте размеры train и validation и баланс классов в каждой части.

Импорты уже есть в блоке «Дано» — повторно импортировать не обязательно, если ячейка «Дано» выполнена.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — выполнен `train_test_split` с `test_size=0.30`.
- **1.0 балл** — указаны `stratify=y` и `random_state=42` (или `RANDOM_STATE`).
- **1.0 балл** — явно сохранены `X_train`, `X_val`, `y_train`, `y_val` и показаны размеры/баланс.

### Снижение баллов

- Нет `stratify` → минус **0.5**.
- Validation не выделена явно (нет `X_val` / `y_val`) → минус **0.5**.
- Изменён блок «Дано» → минус **1.0**.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

print("train:", X_train.shape[0], "| классы:", np.bincount(y_train))
print("val:  ", X_val.shape[0], "| классы:", np.bincount(y_val))


---
## Задание 2. Глубина и переобучение — **4 балла**

Обучите деревья с разной максимальной глубиной: `max_depth` из списка `[1, 3, 6, None]`.

Для каждой модели посчитайте **accuracy** на train и на validation. Соберите результаты в таблицу (`DataFrame` или аккуратный `print`).

Обратите внимание на случаи, где train accuracy почти 1.0, а validation уже не растёт или падает — это типичный сигнал **переобучения** дерева.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучены деревья для всех значений `max_depth` из `[1, 3, 6, None]`.
- **1.0 балл** — посчитана train accuracy для каждого варианта.
- **1.0 балл** — посчитана validation accuracy для каждого варианта.
- **1.0 балл** — результаты собраны в таблицу (или сопоставимый структурированный вывод).

### Снижение баллов

- Сравнение только по train → минус **1.0**.
- Нет таблицы / структурированного сравнения → минус **0.5**.
- Не использован `random_state=RANDOM_STATE` → минус **0.5**.


In [ ]:
depth_rows = []
for d in [1, 3, 6, None]:
    model = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    depth_rows.append(
        {
            "max_depth": "None" if d is None else d,
            "train_acc": round(train_acc, 3),
            "val_acc": round(val_acc, 3),
            "n_leaves": model.get_n_leaves(),
        }
    )

depth_table = pd.DataFrame(depth_rows)
depth_table


---
## Задание 3. Подбор `max_depth` и график — **4 балла**

Переберите `max_depth` от 1 до 12. Для каждой глубины посчитайте validation accuracy.

1. Сохраните лучшую глубину в переменную `best_depth` (выбор **только по validation**).
2. Постройте **график**: по оси X — `max_depth`, по оси Y — accuracy; две линии — train и validation.
3. У графика должны быть заголовок и подписи осей.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — перебраны `max_depth` от 1 до 12.
- **1.0 балл** — для каждой глубины посчитаны train и validation accuracy.
- **1.0 балл** — выбран и сохранён `best_depth` по validation accuracy.
- **1.0 балл** — есть график с заголовком и подписями осей (train и validation на одном рисунке).

### Снижение баллов

- `best_depth` выбран по train → минус **1.0**.
- Нет сохранённого `best_depth` → минус **0.5**.
- График без заголовка или без подписей осей → минус **0.5**.


In [ ]:
depths = list(range(1, 13))
train_scores = []
val_scores = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    val_scores.append(accuracy_score(y_val, model.predict(X_val)))

best_depth = depths[int(np.argmax(val_scores))]
print("best_depth:", best_depth, "| val acc:", round(max(val_scores), 3))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_scores, marker="o", label="train")
plt.plot(depths, val_scores, marker="o", label="validation")
plt.axvline(best_depth, color="gray", linestyle="--", label=f"best_depth={best_depth}")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("Подбор глубины дерева для школьного пропуска")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
## Задание 4. Визуализация дерева (`plot_tree`) — **3 балла**

Дерево удобно читать как **черновик устава**: в каждом узле вопрос «если признак ≤ порога…».

1. Обучите дерево с `max_depth=best_depth` и `random_state=RANDOM_STATE`.
2. Нарисуйте его через `plot_tree` (figsize не меньше 10×6).
3. Передайте `feature_names=FEATURE_NAMES` и `class_names=CLASS_NAMES`, параметр `filled=True`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучено дерево с `best_depth`.
- **1.0 балл** — построена визуализация через `plot_tree`.
- **1.0 балл** — указаны понятные `feature_names` и `class_names`, рисунок читаемый (достаточный `figsize`).

### Снижение баллов

- Визуализирована не выбранная модель / другая глубина без объяснения → минус **0.5**.
- Нет имён признаков или классов → минус **0.5**.
- График нечитаем из-за масштаба → минус **0.5**.


In [ ]:
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
best_tree.fit(X_train, y_train)

plt.figure(figsize=(14, 8))
plot_tree(
    best_tree,
    feature_names=FEATURE_NAMES,
    class_names=CLASS_NAMES,
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title(f"Черновик устава турникета (max_depth={best_depth})")
plt.show()


---
## Задание 5. `min_samples_leaf` — **4 балла**

Чтобы эффект был заметнее, возьмите **глубокое** дерево (`max_depth=None`) и проверьте влияние `min_samples_leaf` на значениях `[1, 5, 15, 30]`.

Для каждого варианта обучите дерево на train и посчитайте train / validation accuracy и число листьев. Сравните результаты в таблице и кратко сформулируйте вывод: увеличение `min_samples_leaf` обычно упрощает дерево и может снизить переобучение, но слишком большое значение ухудшает качество.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — проверены все значения `min_samples_leaf` из `[1, 5, 15, 30]`.
- **1.0 балл** — при сравнении зафиксирован `max_depth=None` (глубокое дерево).
- **1.0 балл** — посчитана validation accuracy для каждого варианта.
- **1.0 балл** — есть сравнение (таблица) и краткий текстовый вывод.

### Снижение баллов

- Одновременно меняются другие параметры без объяснения → минус **0.5**.
- Сравнение только на train → минус **1.0**.
- Нет текстового вывода → минус **0.5**.


In [ ]:
leaf_rows = []
for leaf in [1, 5, 15, 30]:
    model = DecisionTreeClassifier(
        max_depth=None,
        min_samples_leaf=leaf,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    leaf_rows.append(
        {
            "min_samples_leaf": leaf,
            "train_acc": round(accuracy_score(y_train, model.predict(X_train)), 3),
            "val_acc": round(accuracy_score(y_val, model.predict(X_val)), 3),
            "n_leaves": model.get_n_leaves(),
        }
    )

leaf_table = pd.DataFrame(leaf_rows)
print(leaf_table.to_string(index=False))
print(
    "Вывод: рост min_samples_leaf упрощает дерево (меньше листьев) и сужает разрыв train/validation. "
    "Слишком большой лист может ухудшить validation accuracy."
)
leaf_table


---
## Задание 6. Обрезка `ccp_alpha` — **4 балла**

Параметр `ccp_alpha` управляет **обрезкой** дерева после обучения (cost-complexity pruning): чем больше значение, тем сильнее дерево упрощается.

Для полного дерева (`max_depth=None`) переберите `ccp_alpha` в `[0.0, 0.005, 0.02, 0.08]`.

Для каждого значения выведите:
- число листьев (`get_n_leaves()`);
- глубину (`get_depth()`);
- validation accuracy.

Сделайте короткий вывод: умеренная обрезка может помочь, чрезмерная — слишком сильно упрощает устав.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — проверены все значения `ccp_alpha` из списка.
- **1.0 балл** — модели обучены на train с `max_depth=None` (полное дерево + pruning).
- **1.0 балл** — для каждого варианта посчитана validation accuracy (и показаны листья/глубина).
- **1.0 балл** — есть вывод о влиянии pruning (умеренная vs чрезмерная обрезка).

### Снижение баллов

- `ccp_alpha` не меняется в эксперименте → минус **1.0**.
- Нет validation accuracy → минус **0.5**.
- Нет вывода о чрезмерной/умеренной обрезке → минус **0.5**.


In [ ]:
prune_rows = []
for alpha in [0.0, 0.005, 0.02, 0.08]:
    model = DecisionTreeClassifier(
        max_depth=None,
        ccp_alpha=alpha,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    prune_rows.append(
        {
            "ccp_alpha": alpha,
            "n_leaves": model.get_n_leaves(),
            "depth": model.get_depth(),
            "val_acc": round(accuracy_score(y_val, model.predict(X_val)), 3),
        }
    )

prune_table = pd.DataFrame(prune_rows)
print(prune_table.to_string(index=False))
print(
    "Вывод: при росте ccp_alpha листьев становится меньше. "
    "Умеренная обрезка может сохранить качество; слишком большой alpha упрощает дерево чрезмерно."
)
prune_table


---
## Задание 7. Confusion matrix (heatmap) — **3 балла**

Обучите финальное дерево с `max_depth=best_depth` на train и постройте **матрицу ошибок** на validation.

1. Используйте `confusion_matrix` и визуализацию-heatmap через `ConfusionMatrixDisplay`.
2. Подписи классов: `CLASS_NAMES` (`не пустить`, `пустить`).
3. Добавьте заголовок к рисунку.
4. Кратко прокомментируйте: что означает ошибка «модель пустила, а по журналу нельзя» и наоборот.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — финальная модель обучена на train с `best_depth`.
- **1.0 балл** — `confusion_matrix` посчитана на **validation**.
- **1.0 балл** — есть heatmap (`ConfusionMatrixDisplay` или аналог) с понятными подписями классов и заголовком.

### Снижение баллов

- Матрица посчитана на train вместо validation → минус **1.0**.
- Нет heatmap / только «сырой» массив без визуализации → минус **0.5**.
- Неясен порядок классов → минус **0.5**.


In [ ]:
final_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
final_tree.fit(X_train, y_train)
y_val_pred = final_tree.predict(X_val)

cm = confusion_matrix(y_val, y_val_pred)
print("confusion_matrix:\n", cm)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"Матрица ошибок на validation (max_depth={best_depth})")
plt.tight_layout()
plt.show()

print(
    "Комментарий: FP — модель предложила «пустить», а в журнале «не пустить» "
    "(лишний автопроход). FN — модель предложила «не пустить», хотя по журналу можно было пустить "
    "(лишняя ручная проверка у дежурного)."
)


---
## Задание 8. Feature importance — **3 балла**

Выведите `feature_importances_` финального дерева и свяжите важности с именами из `FEATURE_NAMES` (удобно через `Series` или таблицу).

В markdown ниже кратко объясните:
1. какие признаки оказались самыми важными для «устава»;
2. что большая importance **не равна** причинности (признак мог быть лишь удобным для разбиений).

### Подробные критерии (для проверки LLM)

- **1.0 балл** — выведены `feature_importances_` финальной модели.
- **1.0 балл** — важности сопоставлены с именами признаков (таблица / Series).
- **1.0 балл** — есть текстовое объяснение смысла важности и оговорка про причинность.

### Снижение баллов

- Важности без связи с именами признаков → минус **0.5**.
- Нет объяснения, что важность ≠ причинность → минус **0.5**.


In [ ]:
importances = pd.Series(final_tree.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
print(importances.round(3))

plt.figure(figsize=(7, 4))
importances.plot(kind="bar", color="steelblue")
plt.ylabel("importance")
plt.title("Важность признаков в дереве школьного пропуска")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

importances


**Ответ:**

Самыми важными обычно оказываются `has_note` и `late_min` — дерево часто использует справку и минуты опоздания в разбиениях. `weekday` может помогать на средних опозданиях. Признаки вроде `shift`, `temp_c` и `locker_id` чаще слабые или «шумные».

Большая `feature_importance` значит: признак **часто и полезно** участвовал в уменьшении неопределённости в узлах. Это помогает читать модель, но **не доказывает причинность**: важный признак может лишь коррелировать с решением или дублировать другой.


---
## Задание 9. Итог — **2 балла**

Напишите три коротких вывода по результатам практики:

1. про переобучение глубокого дерева;
2. про пользу ограничений (`max_depth`, `min_samples_leaf`, `ccp_alpha`);
3. про интерпретируемость (`plot_tree` и важности признаков) в сценарии школьного пропуска.

### Подробные критерии (для проверки LLM)

- **0.7 балла** — есть вывод про переобучение глубокого дерева (с опорой на train vs validation).
- **0.7 балла** — есть вывод про пользу ограничений глубины / листа / pruning.
- **0.6 балла** — есть вывод про интерпретируемость дерева как «устава» и границ важности признаков.

### Снижение баллов

- Выводы не связаны с результатами практики → минус **0.5**.
- Меньше трёх пунктов → минус **0.5**.


**Итоговые выводы:**

1. Глубокое дерево легко переобучается на журнале пропусков: train accuracy может стать почти идеальной, а validation — хуже. Модель начинает запоминать шум (в том числе `locker_id`), а не устойчивые правила.
2. Ограничения `max_depth`, `min_samples_leaf` и умеренный `ccp_alpha` упрощают дерево и часто улучшают обобщение: «устав» становится короче и стабильнее.
3. `plot_tree` и важности признаков делают модель читаемой как черновик школьных правил, но интерпретацию всё равно нужно сверять с validation: важный признак — не то же самое, что причина решения, а финальный выбор настроек нельзя делать по train.
